# Notebook 05b: Benchmark Orchestrator

**Persona:** ML Engineer / Platform Engineer

**Purpose:** Run multi-step scaled benchmark tests against the Feature Store pipeline.
Each step configures ingestion volume, serving warehouse concurrency, DT refresh
parameters, and OFT query rate — then runs a timed benchmark capturing QPM and
latency percentiles alongside DT refresh history and batch ingestion stats.

**Works in both Snowflake Workspace and local environments.** When running inside
Workspace, worker sessions are created independently using the SPCS OAuth token
for true session-per-thread isolation. No code changes needed.

**Prerequisites:** Run Notebooks 00–03 first (platform setup, feature registration,
ML development, model deployment). The ingestion task, DTs, and OFTs must exist.

**Output:** Console output during the run, plus JSON and Markdown reports saved
to `_internal_development/benchmark_e2e_demo/results/`.

## 1. Setup

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    sys.path.insert(0, ".")
    from feature_definitions.config import get_session
    session = get_session(role="ACCOUNTADMIN")

if "." not in sys.path:
    sys.path.insert(0, ".")

session.sql("USE ROLE ACCOUNTADMIN").collect()

from feature_definitions.config import get_config, is_workspace
from feature_definitions.orchestrator import ScaleStep, run_e2e_test

cfg = get_config("DEV")
print(f"User:      {session.get_current_user()}")
print(f"Role:      {session.get_current_role()}")
print(f"Database:  {cfg['database']}")
print(f"Workspace: {is_workspace()}")

## 2. Configure Scale Steps

Edit the steps below to control the benchmark progression.
Each step runs for `duration_minutes` and adjusts ingestion, serving,
and refresh parameters. 5 minutes per step gives ~5 DT refresh cycles
at a 1-minute target lag.

In [ ]:
steps = [
    ScaleStep(
        name="baseline",
        duration_minutes=5,
        sessions_per_batch=50,
        orders_per_batch=5,
        serving_clusters=1,
        threads_per_cluster=8,
        refresh_clusters=1,
        dt_target_lag="1 minute",
    ),
    ScaleStep(
        name="2x_ingest",
        duration_minutes=5,
        sessions_per_batch=200,
        orders_per_batch=20,
        serving_clusters=1,
        threads_per_cluster=8,
    ),
    ScaleStep(
        name="scale_serving",
        duration_minutes=5,
        sessions_per_batch=200,
        orders_per_batch=20,
        serving_clusters=2,
        threads_per_cluster=8,
    ),
    ScaleStep(
        name="peak",
        duration_minutes=5,
        sessions_per_batch=500,
        orders_per_batch=50,
        serving_clusters=4,
        threads_per_cluster=8,
        refresh_clusters=4,
    ),
]

print(f"Steps: {len(steps)}")
print(f"Total planned: {sum(s.duration_minutes for s in steps)} minutes")
for s in steps:
    print(f"  {s.name}: {s.duration_minutes}m, "
          f"ingest={s.sessions_per_batch}/{s.orders_per_batch}, "
          f"serving={s.serving_clusters}x{s.threads_per_cluster}t")

## 3. Run Benchmark

This cell will run for the total planned duration (~20 minutes for the
default 4 steps at 5 minutes each). Progress and per-step summaries
print to cell output. Results are saved incrementally — if interrupted,
partial results survive in the JSON file.

In [ ]:
results = run_e2e_test(
    session,
    steps,
    env="DEV",
    warmup_seconds=10,
)

## 4. Results Summary

In [ ]:
import pandas as pd

rows = []
for s in results["steps"]:
    c = s["config"]
    b = s["benchmark"]
    p = b["overall_percentiles"]
    thr = c.get("threads_per_cluster", 8) * c.get("serving_clusters", 1)
    rows.append({
        "Step": c["name"],
        "Duration": f"{c['duration_minutes']}m",
        "Ingest": f"{c['sessions_per_batch']}/{c['orders_per_batch']}",
        "Threads": thr,
        "QPM": round(b["qpm"]),
        "p50 (ms)": round(p["p50"], 1),
        "p90 (ms)": round(p["p90"], 1),
        "p95 (ms)": round(p["p95"], 1),
        "p99 (ms)": round(p["p99"], 1),
        "Errors": b["total_errors"],
    })

df = pd.DataFrame(rows)
print("Cross-Step Comparison:")
df

## 5. DT Refresh History

In [ ]:
for i, s in enumerate(results["steps"]):
    snap = s.get("pipeline_snapshot", {})
    rh = snap.get("dt_refresh_history", [])
    ok = [r for r in rh if "ERROR" not in r]
    if ok:
        dt_df = pd.DataFrame(ok)
        cols = [c for c in ["DT", "REFRESH_ACTION", "DURATION_SECONDS",
                            "EXECUTION_MS", "ROWS_INSERTED", "ROWS_DELETED"]
                if c in dt_df.columns]
        print(f"\nStep {i+1}: {s['config']['name']}")
        display(dt_df[cols]) if cols else None

## 6. Batch Ingestion Stats

In [ ]:
for i, s in enumerate(results["steps"]):
    snap = s.get("pipeline_snapshot", {})
    batches = snap.get("batch_stats", [])
    if batches:
        b_df = pd.DataFrame(batches)
        cols = [c for c in ["LOG_ID", "SESSIONS_GENERATED", "EVENTS_GENERATED",
                            "ORDERS_GENERATED", "DURATION_MS"]
                if c in b_df.columns]
        print(f"\nStep {i+1}: {s['config']['name']}")
        display(b_df[cols]) if cols else None

## 7. Output Files

The orchestrator writes two files to
`_internal_development/benchmark_e2e_demo/results/`:

- **JSON** — machine-readable full results (written incrementally)
- **Markdown** — human-readable report with cross-step tables and per-minute windows

The file paths are printed in the run output above.